In [1]:
import random
import pandas as pd
from collections import defaultdict
from datasets import load_dataset

REVIEWS_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl"

N_SAMPLE = 1000000
MAX_SCAN = 5000000
MIN_COUNT = 5

# -----------------------------
# Reservoir sampling
# -----------------------------
reservoir = []

ds_iter = load_dataset(
    "json",
    data_files=REVIEWS_URL,
    split="train",
    streaming=True
)

for i, row in enumerate(ds_iter):
    if i >= MAX_SCAN:
        break

    if i < N_SAMPLE:
        reservoir.append(row)
    else:
        j = random.randint(0, i)
        if j < N_SAMPLE:
            reservoir[j] = row

sample = reservoir

print(f"Scanned: {min(i+1, MAX_SCAN):,}")
print(f"Sampled: {len(sample):,}")

# -----------------------------
# Stat + Filter
# Keep rows with helpful_vote > 0
# and keep only ASINs with count > MIN_COUNT
# -----------------------------
asin_counter = defaultdict(int)
buffer = defaultdict(list)

for row in sample:
    helpful = row.get("helpful_vote", 0)
    if helpful > 0:
        asin = row.get("asin")
        if asin is None:
            continue

        asin_counter[asin] += 1
        buffer[asin].append({
            "images": row.get("images", []),
            "user_id": row.get("user_id"),
            "asin": row.get("asin"),
            "parent_asin": row.get("parent_asin"),
            "rating": row.get("rating"),
            "title": row.get("title", ""),
            "text": row.get("text", ""),
            "timestamp": row.get("timestamp"),
            "verified_purchase": row.get("verified_purchase", False),
            "helpful_vote": row.get("helpful_vote", 0),
        })

filtered = [
    r for asin, cnt in asin_counter.items() if cnt > MIN_COUNT
    for r in buffer[asin]
]

# -----------------------------
# To DataFrame
# -----------------------------
df = pd.DataFrame(filtered)

print(f"Final dataset size: {len(df):,}")
print(f"Unique ASINs: {df['asin'].nunique():,}")
print(df.columns.tolist())
df.head()

Scanned: 5,000,000
Sampled: 1,000,000


Final dataset size: 48,705
Unique ASINs: 4,195
['images', 'user_id', 'asin', 'parent_asin', 'rating', 'title', 'text', 'timestamp', 'verified_purchase', 'helpful_vote']


,images,user_id,asin,parent_asin,rating,title,text,timestamp,verified_purchase,helpful_vote
0,[],AHMXSCFLO5GHIXYRPDNYF6A7IWCQ,B08L6ZYW21,B0C33C3XFV,4.0,"Great sound, quality and price..",I like earphones that are connected as I have ...,1629148161970,True,1
1,[],AHLR2P6ZIIWPKAVAQBHSBM2PFRTQ,B08L6ZYW21,B0C33C3XFV,2.0,Not for me,These buds are not nearly loud enough. And ba...,1632162140196,True,1
2,[],AHWXPUEWRKSB2QLMANPHDQDU4KOQ,B08L6ZYW21,B0C33C3XFV,3.0,Missing simple feature,This product is miss an auto-off feature. My b...,1634873050205,True,1
3,[],AGKANQXLKNHQD7XO2SZJG6MYCERQ,B08L6ZYW21,B0C33C3XFV,3.0,They do not live up to the hype!,I bought these as a backup to to my Bose headp...,1616700645192,True,2
4,[],AEEEKUKP627COLRB7KE4NFO5GCOQ,B08L6ZYW21,B0C33C3XFV,3.0,Budget Friendly Buds,Purchased these for my new iPhone 12. Since t...,1606074278092,True,6


In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# text cleaning
# -----------------------------
df["text"] = df["text"].fillna("")
df["title"] = df["title"].fillna("")

# -----------------------------
# label (binary classification)
# -----------------------------
df["label"] = (df["helpful_vote"] >= 2).astype(int)

# -----------------------------
# Feature Engineering
# -----------------------------

# text lengths
df["text_len"] = df["text"].apply(lambda x: len(x.split()))
df["title_len"] = df["title"].apply(lambda x: len(x.split()))

# punctuation marks usage style
df["exclamation_count"] = df["text"].apply(lambda x: x.count("!"))
df["question_count"] = df["text"].apply(lambda x: x.count("?"))

# has image? 
if "images" in df.columns:
    df["has_image"] = df["images"].apply(
        lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
    )
else:
    df["has_image"] = 0

# verified_purchase
df["verified_purchase"] = df["verified_purchase"].astype(int)

# rating
df["rating"] = df["rating"].astype(float)

# title to text length ratio
df["length_ratio"] = df["title_len"] / (df["text_len"] + 1)

# -----------------------------
# combined text for tokenizer
# -----------------------------
df["combined_text"] = df["title"] + " [SEP] " + df["text"]

# -----------------------------
# Tokenize for DistilBERT
# -----------------------------
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

encodings = tokenizer(
    df["combined_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

# -----------------------------
# numeric features matrix for MLP
# -----------------------------
numeric_features = df[
    [
        "rating",
        "text_len",
        "title_len",
        "exclamation_count",
        "question_count",
        "has_image",
        "verified_purchase",
        "length_ratio",
    ]
].values.astype(np.float32)

# label
labels = df["label"].values

In [3]:
# Train-test split
from sklearn.model_selection import train_test_split

############### FIX THIS LATER################
# i did tokenizing before the split
##############################################

# 70 train 15 val 15 test
# train vs temp
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

# val vs test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

In [4]:
# text encodings
train_encodings = {k: v[train_idx] for k, v in encodings.items()}
val_encodings   = {k: v[val_idx] for k, v in encodings.items()}
test_encodings  = {k: v[test_idx] for k, v in encodings.items()}

# numeric features
train_num = numeric_features[train_idx]
val_num   = numeric_features[val_idx]
test_num  = numeric_features[test_idx]

# standardize numerics
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_num = scaler.fit_transform(train_num)
val_num   = scaler.transform(val_num)
test_num  = scaler.transform(test_num)

# labels
train_labels = labels[train_idx]
val_labels   = labels[val_idx]
test_labels  = labels[test_idx]

In [5]:
import torch
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, encodings, numeric_features, labels):
        self.encodings = encodings
        self.numeric = torch.tensor(numeric_features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["numeric"] = self.numeric[idx]
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [6]:
train_dataset = ReviewDataset(train_encodings, train_num, train_labels)
val_dataset   = ReviewDataset(val_encodings, val_num, val_labels)
test_dataset  = ReviewDataset(test_encodings, test_num, test_labels)

In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [8]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

34093 7306 7306


In [9]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

class DistilBertMLPFusion(nn.Module):
    def __init__(
        self,
        bert_model_name="distilbert-base-uncased",
        numeric_dim=8,
        num_classes=2,
        text_dropout=0.3,
        fusion_hidden_dim=128
    ):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained(bert_model_name)
        bert_hidden_size = self.bert.config.hidden_size

        self.numeric_branch = nn.Sequential(
            nn.Linear(numeric_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bert_hidden_size + 32, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(text_dropout),
            nn.Linear(fusion_hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, numeric_features):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        numeric_out = self.numeric_branch(numeric_features)

        fused = torch.cat([cls_embedding, numeric_out], dim=1)
        logits = self.classifier(fused)
        return logits

In [10]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DistilBertMLPFusion(
    bert_model_name="distilbert-base-uncased",
    numeric_dim=train_num.shape[1],
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Training function

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, acc, f1, precision, recall


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, acc, f1, precision, recall

In [12]:
# training loop

num_epochs = 3
best_val_f1 = 0
best_model_state = None

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1, train_prec, train_rec = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f} | Precision: {train_prec:.4f} | Recall: {train_rec:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | Precision: {val_prec:.4f} | Recall: {val_rec:.4f}")
    print("-" * 80)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

Epoch 1/3
Train Loss: 0.6119 | Acc: 0.6698 | F1: 0.5373 | Precision: 0.6237 | Recall: 0.4719
Val   Loss: 0.6078 | Acc: 0.6700 | F1: 0.5305 | Precision: 0.6285 | Recall: 0.4589
--------------------------------------------------------------------------------


Epoch 2/3
Train Loss: 0.5923 | Acc: 0.6848 | F1: 0.5695 | Precision: 0.6397 | Recall: 0.5132
Val   Loss: 0.6249 | Acc: 0.6753 | F1: 0.5686 | Precision: 0.6178 | Recall: 0.5266
--------------------------------------------------------------------------------


Epoch 3/3
Train Loss: 0.5413 | Acc: 0.7259 | F1: 0.6482 | Precision: 0.6770 | Recall: 0.6217
Val   Loss: 0.6384 | Acc: 0.6518 | F1: 0.5468 | Precision: 0.5801 | Recall: 0.5172
--------------------------------------------------------------------------------


In [13]:
# verify on test

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_loss, test_acc, test_f1, test_prec, test_rec = evaluate(
    model, test_loader, criterion, device
)

print("Best Validation F1:", best_val_f1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f}")
print(f"Test F1:   {test_f1:.4f}")
print(f"Test Prec: {test_prec:.4f}")
print(f"Test Rec:  {test_rec:.4f}")

Best Validation F1: 0.5685703892324482
Test Loss: 0.6189
Test Acc:  0.6755
Test F1:   0.5739
Test Prec: 0.6149
Test Rec:  0.5381


In [14]:
# classification report

from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = get_predictions(model, test_loader, device)

print(classification_report(y_true, y_pred, digits=4))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0     0.7089    0.7695    0.7379      4338
           1     0.6149    0.5381    0.5739      2968

    accuracy                         0.6755      7306
   macro avg     0.6619    0.6538    0.6559      7306
weighted avg     0.6707    0.6755    0.6713      7306

[[3338 1000]
 [1371 1597]]
